In [ ]:
"""
Character‑level Seq2Seq with Teacher Forcing (toy translation)

Dataset:  data/corpus.csv
          each line -> source,target   (tab separated)

Dependencies:
    pip install tensorflow pandas numpy

Author:  Antonis Tziorvas
"""

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.layers import Input, LSTM, Dense, Embedding
from keras.models import Model
from keras.preprocessing.sequence import pad_sequences

In [ ]:
# 1. Load & inspect the data
DATA_PATH = "data/corpus.csv"

if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

# Expecting two columns: source, target (no header)
df = pd.read_csv(DATA_PATH, header=None, names=["en", "fr"], sep="\t")

print(f"Loaded {len(df)} sentence pairs")
df.head(3)

Loaded 100 sentence pairs


,en,fr
0,The cat sits on the mat.,Le chat est assis sur le tapis.
1,I love drinking coffee in the morning.,J’aime boire du café le matin.
2,She bought a new red dress.,Elle a acheté une nouvelle robe rouge.


In [ ]:
# 2. Build character vocabularies
def build_vocab(strings):
    """Return a dict {char: index} with a reserved 0 for padding."""
    chars = sorted(set("".join(strings)))
    # +2: 0 = padding, 1 = start‑of‑sequence (SOS)
    # Separate vocabularies for source and target because the alphabets may differ
    char2idx = {c: i + 2 for i, c in enumerate(chars)}
    char2idx["<PAD>"] = 0
    char2idx["<SOS>"] = 1
    idx2char = {i: c for c, i in char2idx.items()}
    return char2idx, idx2char


en_char2idx, en_idx2char = build_vocab(df["en"].astype(str).tolist())
fr_char2idx, fr_idx2char = build_vocab(df["fr"].astype(str).tolist())

en_vocab_size = len(en_char2idx)
fr_vocab_size = len(fr_char2idx)

print(f"Source vocab size (incl. PAD & SOS): {en_vocab_size}")
print(f"Target vocab size (incl. PAD & SOS): {fr_vocab_size}")

Source vocab size (incl. PAD & SOS): 47
Target vocab size (incl. PAD & SOS): 56


In [13]:
# 3. Convert strings to integer sequences
def encode_string(s, char2idx, add_sos=False):
    """Map characters to integers, optionally prepend SOS token."""
    seq = [char2idx.get(ch, 0) for ch in s]  # unknown chars become 0 (PAD)
    if add_sos:
        seq = [char2idx["<SOS>"]] + seq
    return seq


# Encode source strings (no SOS needed for encoder)
encoder_input_seqs = [encode_string(s, en_char2idx) for s in df["en"]]

# For the decoder we need two versions:
#   - decoder_input  :  <SOS> + target
#   - decoder_target :  target + <PAD> (the model tries to predict this)
decoder_input_seqs = [encode_string(t, fr_char2idx, add_sos=True) for t in df["fr"]]
decoder_target_seqs = [
    encode_string(t, fr_char2idx) + [fr_char2idx["<PAD>"]] for t in df["fr"]
]

In [ ]:
# 4. Pad sequences to uniform length

MAX_ENCODER_SEQ_LENGTH = max(len(seq) for seq in encoder_input_seqs)
MAX_DECODER_SEQ_LENGTH = max(len(seq) for seq in decoder_input_seqs)

print("Maximum encoder sequence length :", MAX_ENCODER_SEQ_LENGTH)
print("Maximum decoder sequence length :", MAX_DECODER_SEQ_LENGTH)

encoder_input_data = pad_sequences(
    encoder_input_seqs, maxlen=MAX_ENCODER_SEQ_LENGTH, padding="post"
)

decoder_input_data = pad_sequences(
    decoder_input_seqs, maxlen=MAX_DECODER_SEQ_LENGTH, padding="post"
)

decoder_target_data = pad_sequences(
    decoder_target_seqs, maxlen=MAX_DECODER_SEQ_LENGTH, padding="post"
)
# Teacher forcing: we will feed decoder_input_data during training,
# and ask the network to predict decoder_target_data.

# Question: why convert to one-hot vectors?
decoder_target_onehot = tf.keras.utils.to_categorical(
    decoder_target_data, num_classes=fr_vocab_size
)

print("Shapes:")
print(" encoder_input_data :", encoder_input_data.shape)
print(" decoder_input_data :", decoder_input_data.shape)
print(" decoder_target_onehot :", decoder_target_onehot.shape)

Maximum encoder sequence length : 43
Maximum decoder sequence length : 55
Shapes:
 encoder_input_data : (100, 43)
 decoder_input_data : (100, 55)
 decoder_target_onehot : (100, 55, 56)


In [ ]:
# 5. Build the Seq2Seq model (Encoder‑Decoder with LSTM)

EMBEDDING_DIM = 32
LATENT_DIM = 128  # number of LSTM units

# Encoder
# ...

encoder_inputs = ...

# Decoder
# ...

decoder_inputs = ...
decoder_outputs = ...

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()

In [ ]:
# Encoder
encoder_inputs = Input(shape=(None,), name="encoder_inputs")
enc_emb = Embedding(
    input_dim=en_vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,  # ignore PAD tokens in the LSTM
    name="encoder_embedding",
)(encoder_inputs)

encoder_lstm = LSTM(LATENT_DIM, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]  # will be passed to the decoder

# Decoder
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
dec_emb = Embedding(
    input_dim=fr_vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="decoder_embedding",
)(decoder_inputs)

decoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm",
)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# Final dense layer to predict next character
decoder_dense = Dense(fr_vocab_size, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

# Full training model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()

In [22]:
# 6. Train with teacher forcing

EPOCHS = 30
BATCH_SIZE = 5

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_onehot,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,
)

Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.1391 - loss: 3.7896 - val_accuracy: 0.1437 - val_loss: 3.2003
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.1381 - loss: 3.1331 - val_accuracy: 0.1437 - val_loss: 2.9614
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - accuracy: 0.1559 - loss: 3.0206 - val_accuracy: 0.1585 - val_loss: 2.8894
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.1652 - loss: 2.9609 - val_accuracy: 0.1585 - val_loss: 2.8277
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.1992 - loss: 2.9036 - val_accuracy: 0.2444 - val_loss: 2.7722
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.2210 - loss: 2.8445 - val_accuracy: 0.2474 - val_loss: 2.7000
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - accuracy: 0.2489 - loss: 2.7738 - val_accuracy: 0.2844 - val_loss: 2.6293
Epoch 8/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - accuracy: 0.2740 - loss: 2.6962 - val_accuracy: 0.2948 - v

In [23]:
# 7. Inference models (encoder + decoder without teacher forcing)

# Encoder model: maps input sequence -> states
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder model: takes one timestep at a time + states, returns next char + new states
decoder_state_input_h = Input(shape=(LATENT_DIM,), name="dec_state_h")
decoder_state_input_c = Input(shape=(LATENT_DIM,), name="dec_state_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = decoder_embedding_layer = model.get_layer("decoder_embedding")(
    decoder_inputs
)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2,
)

In [ ]:
# 8. Decoding function (greedy search)


def decode_sequence(input_seq, max_target_len=MAX_DECODER_SEQ_LENGTH):
    """
    Greedy decoding: feed the decoder one character at a time.
    input_seq must be a 2‑D array (1, seq_len) already padded.
    """
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Start token
    target_seq = np.array([[fr_char2idx["<SOS>"]]])

    # Collect generated characters
    decoded_chars = []

    for _ in range(max_target_len):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        # Sample the most probable token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = fr_idx2char.get(sampled_token_index, "")

        if sampled_char == "<PAD>" or sampled_char == "":
            break

        decoded_chars.append(sampled_char)

        # Update the target sequence (of length 1)
        target_seq = np.array([[sampled_token_index]])

        # Update states
        states_value = [h, c]

    return "".join(decoded_chars)

In [25]:
# 9. Quick demo on a few test sentences

print("\n--- Demo translations ---")
for i in range(5):
    en_sentence = df["en"].iloc[i]
    # Encode source
    en_seq = encode_string(en_sentence, en_char2idx)
    en_seq = pad_sequences([en_seq], maxlen=MAX_ENCODER_SEQ_LENGTH, padding="post")
    translation = decode_sequence(en_seq)
    print(f"Source    : {en_sentence}")
    print(f"Target GT : {df['fr'].iloc[i]}")
    print(f"Predicted : {translation}")
    print("-" * 40)


--- Demo translations ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━

In [ ]:
# 10. Save models (optional)

model.save("seq2seq_char_teacherforcing.keras")
encoder_model.save("seq2seq_char_encoder.keras")
decoder_model.save("seq2seq_char_decoder.keras")
print("\nModels saved to disk.")